In [ ]:
# import the required libraries
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/gpt2_chunk_4.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,523670,"Documentary, History, TV Movie",tt6394146,fr,Le MystÃ¨re de la mort d'Hitler,"On May 2, 1945, Soviets take control over the ...",1.400,3/20/2018,"German, English, French, Russian",7.0,2.0,523670.0,based on novel or book investigation forensic ...,"[11, 15, 14]",-1.18988156e-01 -2.32806597e-02 -8.30981731e-0...,1.32116228e-01 -3.46202344e-01 2.79474169e-0...
1,False,523671,"Drama, TV Movie",tt0073020,en,Friendly Persuasion,"During the Civil War, a Quaker couple risks th...",1.400,5/18/1975,English,0.0,1.0,523671.0,slavery american civil war underground railroa...,"[8, 14]",-3.61987323e-01 -2.77490973e-01 -3.52308601e-0...,3.55162919e-01 2.11510900e-02 -1.23051107e-0...
2,False,523672,Documentary,tt5466630,fr,"La dette, une spirale infernale?",NaN,0.600,2/3/2015,"German, English, French",0.0,0.0,523672.0,economy,[11],-4.51944880e-02 -8.34512785e-02 -2.81697214e-0...,1.87571898e-01 -4.41325873e-01 2.05782731e-0...
3,False,523676,Documentary,tt4162484,fr,Pirat@ge,NaN,0.600,4/15/2011,"German, English, French",7.5,1.0,523676.0,hacker internet,[11],-1.92522407e-02 -1.22282542e-01 -3.14873867e-0...,-2.72318274e-02 -2.51108587e-01 6.07782602e-0...
4,False,523677,Documentary,tt6672398,fr,Tattoos: Tous tatouÃ©s !,How did they migrate from the margin to the ma...,0.600,11/16/2013,"English, French",0.0,0.0,523677.0,tattoo tatouage,[11],1.22811779e-01 -1.46228239e-01 -2.09712893e-0...,-1.16580293e-01 -6.62005246e-02 2.20130384e-0...
5,False,523679,"Comedy, Drama",tt4498818,fa,Ø®ÙˆØ§Ø¨ Ø²Ø¯Ù‡ Ù‡Ø§,An old rich man dreams of his deceased mother....,0.856,5/11/2013,Persian,0.0,0.0,523679.0,khabzadeha Ø®ÙˆØ§Ø¨ Ø²Ø¯Ù‡ Ù‡Ø§,"[5, 8]",-6.09422803e-01 -2.70402610e-01 -3.50635350e-0...,2.70503670e-01 -1.18201084e-01 -1.18996143e-0...
6,False,523680,"Documentary, TV Movie",tt8391060,en,Prince Harry's Story: Four Royal Weddings,This documentary charts Harry's journey throug...,0.723,5/3/2018,English,8.8,3.0,523680.0,palace intrigue royal wedding british monarchy,"[11, 14]",-2.22176127e-02 -2.89984718e-02 -4.97691780e-0...,1.07055821e-01 8.49735737e-02 1.52011529e-0...
7,False,523685,Animation,NaN,en,The Falling Sky,Refashioning the original intention of footage...,0.600,6/10/2018,[],0.0,0.0,523685.0,woman director short film,[9],1.98325992e-01 2.17532739e-01 -9.27819788e-0...,-2.02643648e-01 -5.95186412e-01 -5.67901134e-0...
8,False,523687,Music,NaN,en,Nas: Live at Reading and Leeds Festival 2016,NaN,0.623,NaN,English,10.0,1.0,523687.0,concert hip-hop,[18],2.96507776e-03 -7.48196542e-02 -6.74719810e-0...,-1.88246697e-01 -3.44341919e-02 -1.55293807e-0...
9,False,523704,Comedy,tt7607318,es,Carlos Ballarta: furia Ã±era,Mexican comic Carlos Ballarta mocks himself an...,2.136,5/11/2018,Spanish,7.0,20.0,523704.0,stand-up comedy,[5],-6.52223676e-02 1.68744475e-02 -3.78949761e-0...,-1.05920220e-02 -4.50430006e-01 1.09621398e-0...


In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,523670,tt6394146,Le MystÃ¨re de la mort d'Hitler,"On May 2, 1945, Soviets take control over the ..."
1,523671,tt0073020,Friendly Persuasion,"During the Civil War, a Quaker couple risks th..."
2,523672,tt5466630,"La dette, une spirale infernale?",NaN
3,523676,tt4162484,Pirat@ge,NaN
4,523677,tt6672398,Tattoos: Tous tatouÃ©s !,How did they migrate from the margin to the ma...


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1024.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained gpt-2 tokenizer and model
model_name='gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2Model.from_pretrained(model_name)

def create_word_embeddings(sentence):
    if pd.isna(sentence):
      return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'overview_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[-2.61418551e-01  2.20993638e-01  4.64379452e-02  5.98941505e-01\n  2.18234584e-01 -5.07454336e-01  5.40878153e+00 -5.91726638e-02\n -2.12462351e-01 -2.15432793e-02  2.20353618e-01 -2.99488634e-01\n  1.60585448e-01 -3.45377512e-02 -1.11695968e-01  3.06225121e-01\n  5.37976146e-01  1.47365540e-01 -6.67109489e-02 -4.52763230e-01\n  3.30340378e-02  9.75887924e-02  7.98002928e-02  3.46329585e-02\n  1.77975863e-01  5.84321693e-02  1.13968901e-01  2.95582980e-01\n -1.56739563e-01  2.94480860e-01  2.77427763e-01 -2.49036234e-02\n -8.74587987e-03  7.97572080e-03 -2.26591513e-01 -1.77095056e-01\n  6.65699997e+01 -2.43608877e-01  1.54910644e-03  5.30091897e-02\n -1.28269359e-03 -1.80742517e-01  6.82189986e-02 -4.48075607e-02\n -1.60092428e-01 -4.24729049e-01  4.07688767e-01  1.10821649e-01\n -2.42309377e-01  3.82700235e-01 -1.69501200e-01  1.19427517e-01\n -2.01701969e-02  2.75239777e-02  1.92939460e-01  9.79596376e-01\n -3.12457383e-01  1.03899330e-01 -7.59520382e-02 -1.57261804e-01\n -1.31649

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '').str.replace(',', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

'-2.61418551e-01  2.20993638e-01  4.64379452e-02  5.98941505e-01  2.18234584e-01 -5.07454336e-01  5.40878153e+00 -5.91726638e-02 -2.12462351e-01 -2.15432793e-02  2.20353618e-01 -2.99488634e-01  1.60585448e-01 -3.45377512e-02 -1.11695968e-01  3.06225121e-01  5.37976146e-01  1.47365540e-01 -6.67109489e-02 -4.52763230e-01  3.30340378e-02  9.75887924e-02  7.98002928e-02  3.46329585e-02  1.77975863e-01  5.84321693e-02  1.13968901e-01  2.95582980e-01 -1.56739563e-01  2.94480860e-01  2.77427763e-01 -2.49036234e-02 -8.74587987e-03  7.97572080e-03 -2.26591513e-01 -1.77095056e-01  6.65699997e+01 -2.43608877e-01  1.54910644e-03  5.30091897e-02 -1.28269359e-03 -1.80742517e-01  6.82189986e-02 -4.48075607e-02 -1.60092428e-01 -4.24729049e-01  4.07688767e-01  1.10821649e-01 -2.42309377e-01  3.82700235e-01 -1.69501200e-01  1.19427517e-01 -2.01701969e-02  2.75239777e-02  1.92939460e-01  9.79596376e-01 -3.12457383e-01  1.03899330e-01 -7.59520382e-02 -1.57261804e-01 -1.31649926e-01 -5.24006225e-02  2.5129

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,523670,tt6394146,Le MystÃ¨re de la mort d'Hitler,"On May 2, 1945, Soviets take control over the ...",-2.61418551e-01 2.20993638e-01 4.64379452e-0...
1,523671,tt0073020,Friendly Persuasion,"During the Civil War, a Quaker couple risks th...",-2.31385842e-01 -1.27544135e-01 -6.72115162e-0...
2,523672,tt5466630,"La dette, une spirale infernale?",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...
3,523676,tt4162484,Pirat@ge,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...
4,523677,tt6672398,Tattoos: Tous tatouÃ©s !,How did they migrate from the margin to the ma...,-1.30844265e-02 -2.27189027e-02 -3.66461396e-0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/old_final_mkg_dataset_gpt2.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,523670,tt6394146,Le MystÃ¨re de la mort d'Hitler,"On May 2, 1945, Soviets take control over the ...",-2.61418551e-01 2.20993638e-01 4.64379452e-0...,False,"Documentary, History, TV Movie",fr,"On May 2, 1945, Soviets take control over the ...",1.4,3/20/2018,"German, English, French, Russian",7.0,2.0,523670.0,based on novel or book investigation forensic ...,"[11, 15, 14]",-1.18988156e-01 -2.32806597e-02 -8.30981731e-0...,1.32116228e-01 -3.46202344e-01 2.79474169e-0...
1,523671,tt0073020,Friendly Persuasion,"During the Civil War, a Quaker couple risks th...",-2.31385842e-01 -1.27544135e-01 -6.72115162e-0...,False,"Drama, TV Movie",en,"During the Civil War, a Quaker couple risks th...",1.4,5/18/1975,English,0.0,1.0,523671.0,slavery american civil war underground railroa...,"[8, 14]",-3.61987323e-01 -2.77490973e-01 -3.52308601e-0...,3.55162919e-01 2.11510900e-02 -1.23051107e-0...
2,523672,tt5466630,"La dette, une spirale infernale?",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Documentary,fr,NaN,0.6,2/3/2015,"German, English, French",0.0,0.0,523672.0,economy,[11],-4.51944880e-02 -8.34512785e-02 -2.81697214e-0...,1.87571898e-01 -4.41325873e-01 2.05782731e-0...
3,523676,tt4162484,Pirat@ge,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Documentary,fr,NaN,0.6,4/15/2011,"German, English, French",7.5,1.0,523676.0,hacker internet,[11],-1.92522407e-02 -1.22282542e-01 -3.14873867e-0...,-2.72318274e-02 -2.51108587e-01 6.07782602e-0...
4,523677,tt6672398,Tattoos: Tous tatouÃ©s !,How did they migrate from the margin to the ma...,-1.30844265e-02 -2.27189027e-02 -3.66461396e-0...,False,Documentary,fr,How did they migrate from the margin to the ma...,0.6,11/16/2013,"English, French",0.0,0.0,523677.0,tattoo tatouage,[11],1.22811779e-01 -1.46228239e-01 -2.09712893e-0...,-1.16580293e-01 -6.62005246e-02 2.20130384e-0...


In [ ]:
merged_df.shape[0]

46946

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

46946

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/final_mkg_dataset_gpt2_chunk_4.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,523670,tt6394146,Le MystÃ¨re de la mort d'Hitler,"On May 2, 1945, Soviets take control over the ...",-2.61418551e-01 2.20993638e-01 4.64379452e-0...,False,"Documentary, History, TV Movie",fr,"On May 2, 1945, Soviets take control over the ...",1.4,3/20/2018,"German, English, French, Russian",7.0,2.0,523670.0,based on novel or book investigation forensic ...,"[11, 15, 14]",-1.18988156e-01 -2.32806597e-02 -8.30981731e-0...,1.32116228e-01 -3.46202344e-01 2.79474169e-0...
1,523671,tt0073020,Friendly Persuasion,"During the Civil War, a Quaker couple risks th...",-2.31385842e-01 -1.27544135e-01 -6.72115162e-0...,False,"Drama, TV Movie",en,"During the Civil War, a Quaker couple risks th...",1.4,5/18/1975,English,0.0,1.0,523671.0,slavery american civil war underground railroa...,"[8, 14]",-3.61987323e-01 -2.77490973e-01 -3.52308601e-0...,3.55162919e-01 2.11510900e-02 -1.23051107e-0...
2,523672,tt5466630,"La dette, une spirale infernale?",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Documentary,fr,NaN,0.6,2/3/2015,"German, English, French",0.0,0.0,523672.0,economy,[11],-4.51944880e-02 -8.34512785e-02 -2.81697214e-0...,1.87571898e-01 -4.41325873e-01 2.05782731e-0...
3,523676,tt4162484,Pirat@ge,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Documentary,fr,NaN,0.6,4/15/2011,"German, English, French",7.5,1.0,523676.0,hacker internet,[11],-1.92522407e-02 -1.22282542e-01 -3.14873867e-0...,-2.72318274e-02 -2.51108587e-01 6.07782602e-0...
4,523677,tt6672398,Tattoos: Tous tatouÃ©s !,How did they migrate from the margin to the ma...,-1.30844265e-02 -2.27189027e-02 -3.66461396e-0...,False,Documentary,fr,How did they migrate from the margin to the ma...,0.6,11/16/2013,"English, French",0.0,0.0,523677.0,tattoo tatouage,[11],1.22811779e-01 -1.46228239e-01 -2.09712893e-0...,-1.16580293e-01 -6.62005246e-02 2.20130384e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input gpt2 embedded dataset chunk-4
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,523670,-2.61418551e-01 2.20993638e-01 4.64379452e-0...,"Documentary, History, TV Movie",3/20/2018,"[11, 15, 14]",-1.18988156e-01 -2.32806597e-02 -8.30981731e-0...,1.32116228e-01 -3.46202344e-01 2.79474169e-0...
1,523671,-2.31385842e-01 -1.27544135e-01 -6.72115162e-0...,"Drama, TV Movie",5/18/1975,"[8, 14]",-3.61987323e-01 -2.77490973e-01 -3.52308601e-0...,3.55162919e-01 2.11510900e-02 -1.23051107e-0...
2,523672,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Documentary,2/3/2015,[11],-4.51944880e-02 -8.34512785e-02 -2.81697214e-0...,1.87571898e-01 -4.41325873e-01 2.05782731e-0...
3,523676,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Documentary,4/15/2011,[11],-1.92522407e-02 -1.22282542e-01 -3.14873867e-0...,-2.72318274e-02 -2.51108587e-01 6.07782602e-0...
4,523677,-1.30844265e-02 -2.27189027e-02 -3.66461396e-0...,Documentary,11/16/2013,[11],1.22811779e-01 -1.46228239e-01 -2.09712893e-0...,-1.16580293e-01 -6.62005246e-02 2.20130384e-0...


In [ ]:
merged_df.shape[0]

46946

In [ ]:
# export the embedded keywords, title and overview dataset (chunk-4)
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_4.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_4.csv")

In [ ]:
test_df.shape[0]

46946

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,523670,-2.61418551e-01 2.20993638e-01 4.64379452e-0...,"Documentary, History, TV Movie",3/20/2018,"[11, 15, 14]",-1.18988156e-01 -2.32806597e-02 -8.30981731e-0...,1.32116228e-01 -3.46202344e-01 2.79474169e-0...
1,523671,-2.31385842e-01 -1.27544135e-01 -6.72115162e-0...,"Drama, TV Movie",5/18/1975,"[8, 14]",-3.61987323e-01 -2.77490973e-01 -3.52308601e-0...,3.55162919e-01 2.11510900e-02 -1.23051107e-0...
2,523672,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Documentary,2/3/2015,[11],-4.51944880e-02 -8.34512785e-02 -2.81697214e-0...,1.87571898e-01 -4.41325873e-01 2.05782731e-0...
3,523676,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Documentary,4/15/2011,[11],-1.92522407e-02 -1.22282542e-01 -3.14873867e-0...,-2.72318274e-02 -2.51108587e-01 6.07782602e-0...
